In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm
import json
import os
import random

In [2]:
with open("../datas/couplet/vocabs", "r") as f:
    vocab = f.read().split('\n')
with open('../datas/couplet/char2id.json', 'r', encoding='utf-8') as f:
    char2id = json.load(f)
with open('../datas/couplet/id2char.json', 'r', encoding='utf-8') as f:
    id2char = json.load(f)
print(len(vocab))
print(len(char2id))
print(len(id2char))

9132
9132
9132


In [3]:
with open('../datas/couplet/train/out_id.json', 'r', encoding='utf-8') as f:
    train_out_id = json.load(f)
with open('../datas/couplet/train/in_id.json', 'r', encoding='utf-8') as f:
    train_in_id = json.load(f)
with open('../datas/couplet/test/out_id.json', 'r', encoding='utf-8') as f:
    test_out_id = json.load(f)
with open('../datas/couplet/test/in_id.json', 'r', encoding='utf-8') as f:
    test_in_id = json.load(f)
with open('../datas/couplet/train/true_len.json', 'r', encoding='utf-8') as f:
    train_true_len = json.load(f)
with open('../datas/couplet/test/true_len.json', 'r', encoding='utf-8') as f:
    test_true_len = json.load(f)
print(type(train_true_len))

<class 'list'>


In [4]:
class Encoder(nn.Module):
	def __init__(self,vocab_size,embed_dim,hidden_size,num_layers,dropout,bidirectional =False):
		super(Encoder, self).__init__()
		self.bidirectional = bidirectional
		self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=0)
		self.rnn = nn.GRU(embed_dim,hidden_size,num_layers,batch_first=True,bidirectional=bidirectional,dropout=dropout)
		self.dropout = nn.Dropout(dropout)
	def forward(self,x,true_len=None):
		if true_len is None:
			true_len = torch.full((x.size(0),), x.size(1),dtype=torch.long)
		embedded = self.dropout(self.embedding(x))
		true_len = true_len.clamp(max=x.size(1))
		packed = pack_padded_sequence(embedded,true_len.cpu(),batch_first=True,enforce_sorted=False)
		output, hidden = self.rnn(packed)
		output, _ = pad_packed_sequence(output,batch_first=True,total_length=x.size(1),padding_value=0)
		if self.bidirectional == True:
			hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
		else:
			hidden = hidden[-1]
		return output, hidden

In [5]:
# 编码器案例
encoder = Encoder(100,128,256,3,0.3,bidirectional=True)
token_ids = torch.randint(0, 20, size=(20,30))
true_len = torch.randint(1, 10, size=(20,))
output, hidden = encoder(token_ids,true_len)
print(output.size())
print(hidden.size())


torch.Size([20, 30, 512])
torch.Size([20, 512])


In [6]:
def attention(q,k):
	""""
	:param q[bs,1,hidden]:
	:param k[bs,t,hidden]:
	:return[bs,t,hidden]:
	"""
	v = k
	score = torch.bmm(q, k.transpose(1,2)) #[bs,1,t]
	alpha = torch.softmax(score,dim=-1) #[bs,1,t]
	value = torch.bmm(alpha,v) #[bs,1,e]
	return value

In [7]:
class Decoder(nn.Module):
	def __init__(self,vocab_size,embed_dim,hidden_size,num_layers,dropout,encoder_bidirectional,device,teacher_forcing_ratio=0.5,encoder_hidden_size=None):
		super(Decoder, self).__init__()
		encoder_directions = 2 if encoder_bidirectional else 1
		self.hidden_size = hidden_size
		self.num_layers = num_layers
		self.device = device
		self.teacher_forcing_ratio = teacher_forcing_ratio
		if encoder_hidden_size is None:
			encoder_hidden_size = hidden_size
		self.attention = attention
		self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=0)
		self.rnn = nn.GRU(embed_dim+encoder_hidden_size,hidden_size,num_layers,batch_first=True,dropout=dropout,bidirectional=False)
		self.init_fc = nn.Linear(encoder_hidden_size * encoder_directions, hidden_size * num_layers,bias=False)
		self.init_fc2 = nn.Linear(encoder_hidden_size * encoder_directions, hidden_size,bias=False)
		self.fc_out = nn.Linear(hidden_size,vocab_size)
		self.dropout = nn.Dropout(dropout)

	def init(self,enc_hidden,enc_output):
		hidden = self.init_fc(enc_hidden)
		output = self.init_fc2(enc_output)
		hidden = hidden.reshape(hidden.size(0),self.num_layers,self.hidden_size).permute(1,0,2)
		return hidden.contiguous(),output

	def forward(self,x,enc_hidden,enc_output):
		batch_size = x.shape[0]
		x_len = x.shape[1]
		x_vocab_size = self.fc_out.out_features
		dec_hidden = enc_hidden

		outputs = torch.zeros(batch_size, x_len, x_vocab_size).to(self.device)
		input = x[:,0].unsqueeze(1) # x[bs,1]
		for t in range(1,x_len):
			embedded = self.dropout(self.embedding(input)) # x[bs,1,hidden]
			atten_value = self.attention(dec_hidden[-1].unsqueeze(1),enc_output)
			rnn_input = torch.cat((embedded,atten_value),dim=2)
			dec_output, dec_hidden = self.rnn(rnn_input,dec_hidden)
			dec_score = self.fc_out(dec_output).squeeze(1) # [bs,vocab_size]
			outputs[:,t,:] = dec_score # [bs,vocab_size]
			top1 = dec_score.argmax(-1) # [bs,]
			teacher_force = random.random() < self.teacher_forcing_ratio
			input = x[:,t].unsqueeze(1) if teacher_force else top1.unsqueeze(1) # [bs,1]
		return outputs, dec_hidden

	def forward_step(self, input, dec_hidden, enc_output):
	    embedded = self.dropout(self.embedding(input))
	    atten_value = self.attention(dec_hidden[-1].unsqueeze(1), enc_output)
	    rnn_input = torch.cat((embedded, atten_value), dim=2)
	    dec_output, dec_hidden = self.rnn(rnn_input, dec_hidden)
	    dec_score = self.fc_out(dec_output)  # [bs, 1, vocab_size]
	    return dec_score, dec_hidden

In [8]:
# 解码器案例
decoder = Decoder(100,128,256,2,0.3,True,'cpu')
token_ids = torch.randint(0, 20, size=(20,20),dtype=torch.long) # 解码器的输入
enc_hidden = torch.rand(20,256*2) # 编码器提取出来的特征向量
enc_output = torch.rand(20,20,256*2)
decoder.train()
enc_hidden,enc_output = decoder.init(enc_hidden,enc_output)
decoder_score,hidden = decoder(token_ids, enc_hidden, enc_output)
print(f"解码器输出:{decoder_score.size()}")
print(f"解码器输出:{hidden.size()}")


解码器输出:torch.Size([20, 20, 100])
解码器输出:torch.Size([2, 20, 256])


In [9]:
class Seq2Seq(nn.Module):
	def __init__(self,encoder,decoder,device,teacher_forcing_ratio=0.5):
		super(Seq2Seq, self).__init__()
		self.encoder = encoder
		self.decoder = decoder
		self.device = device
		self.attention = attention
		self.teacher_forcing_ratio = teacher_forcing_ratio

	def forward(self,x,y,x_true_len=None):
		enc_output, enc_hidden = self.encoder(x,x_true_len)
		enc_hidden, enc_output = self.decoder.init(enc_hidden,enc_output)
		outputs, _ = self.decoder(y,enc_hidden,enc_output)
		return outputs

	@torch.no_grad()
	def generate(self,x,sos_id,eos_id,x_true_len=None,max_len=50):
		bs = x.size(0)
		enc_output, hidden = self.encoder(x, x_true_len)
		hidden,enc_output = self.decoder.init(hidden,enc_output)
		# 起始 token
		inp = torch.full((bs,1),sos_id,dtype=torch.long).to(self.device)
		results = []
		finished = torch.zeros(bs,dtype=torch.bool).to(self.device)
		for _ in range(max_len):
			output,hidden = self.decoder.forward_step(inp,hidden,enc_output)
			pred = output[:,0,:].argmax(dim=-1) # [bs,1,vocab_size]
			pred = pred.unsqueeze(1) #[bs,]
			results.append(pred)
			finished |= (pred.squeeze(1) == eos_id)
			if finished.all():
				break
			inp = pred
		return torch.cat(results,dim=1)

In [10]:
# seq2seq案例
encoder = Encoder(100,128,256,3,0.3,bidirectional=True)
decoder = Decoder(100,128,256,3,0.3,True,'cpu')

seq2seq = Seq2Seq(encoder,decoder,torch.device('cpu'),teacher_forcing_ratio=0.5)

train_ids = torch.randint(0, 20, size=(20,20))
test_ids = torch.randint(0, 20, size=(20,20))

score = seq2seq(token_ids,test_ids)
print(f"输出:{score.shape}")
output = seq2seq.generate(test_ids,1,2,max_len=20)
print(output.size())

输出:torch.Size([20, 20, 100])
torch.Size([20, 20])


In [11]:
class MyDataset(Dataset):
    def __init__(self, data, target, true_lens):

        data = np.array(data).astype('int64')
        target = np.array(target).astype('int64')
        true_lens = np.array(true_lens).astype('int64')

        self.data = torch.from_numpy(data)
        self.target = torch.from_numpy(target)
        self.true_lens = torch.from_numpy(true_lens)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx], self.target[idx],self.true_lens[idx]

In [12]:
print(len(train_in_id))
print(len(test_in_id))
train_in_id = train_in_id[:500]
train_out_id = train_out_id[:500]
test_in_id = test_in_id[:100]
test_out_id = test_out_id[:100]

770492
4001


In [13]:
vocab_size = len(vocab)
embed_dim = 128
hidden_size = 256
num_layers = 2
bidirectional = True
dropout = 0.3
teacher_forcing_ratio = 0.5
batch_size = 64
learning_rate = 0.001
epochs = 5
seed = 24
random.seed(seed)
torch.manual_seed(seed)
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(device)

mps


In [14]:
X_train,y_train,X_test,y_test = train_in_id,train_out_id,test_in_id,test_out_id
ture_lens_train,ture_lens_test = train_true_len,test_true_len
train_loader = DataLoader(MyDataset(X_train,y_train,ture_lens_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(MyDataset(X_test,y_test,ture_lens_test), batch_size=batch_size//2, shuffle=False)

In [15]:
encoder = Encoder(vocab_size,embed_dim,hidden_size,num_layers,dropout,bidirectional).to(device)
decoder = Decoder(vocab_size,embed_dim,hidden_size,num_layers,dropout,bidirectional,device).to(device)
model = Seq2Seq(encoder,decoder,device,teacher_forcing_ratio).to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=0).to(device)

In [16]:
summary_dir = "../datas/couplet/summary/v2"
model_dir = "../datas/couplet/model/v2"
best_model_path = os.path.join(model_dir,"best_model.pkl")
last_model_path = os.path.join(model_dir,"last_model.pkl")
os.makedirs(model_dir,exist_ok=True)
os.makedirs(summary_dir,exist_ok=True)

writer = SummaryWriter(log_dir=summary_dir)
start_epoch = 0
best_acc = 0.0
resume_training = True

if os.path.exists(last_model_path) and resume_training:
	print(f"发现历史保存模型 '{last_model_path}'，正在加载...")
	checkpoint = torch.load(last_model_path,map_location=device,weights_only=False)
	model.load_state_dict(checkpoint['model'])
	optimizer.load_state_dict(checkpoint['optimizer'])
	start_epoch = checkpoint["epoch"] + 1
	best_acc = checkpoint['best_acc']
	print(f"成功恢复训练，将从 Epoch {start_epoch} 开始，历史最佳准确率: {best_acc:.2f}%\n")

for epoch in range(start_epoch,start_epoch+epochs):
	teacher_forcing_ratio = max(0.1, 0.5 - epoch * 0.08)
	model.decoder.teacher_forcing_ratio = teacher_forcing_ratio

	model.train()
	train_loss_sum = 0.0

	train_pbar = tqdm(train_loader, desc=f'Train Epoch {epoch}/{start_epoch+epochs-1}')
	for batch_idx, (data, target, true_len) in enumerate(train_pbar):
		data,target,true_len = data.to(device), target.to(device),true_len.cpu()
		optimizer.zero_grad()
		output = model(data,target,true_len)
		loss = criterion(output[:, 1:, :].permute(0,2,1), target[:, 1:])
		loss.backward()

		torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
		optimizer.step()

		train_loss_sum += loss.item()
		train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
		global_step = epoch * len(train_loader) + batch_idx
	# writer.add_scalar('Train/Step_Loss', loss.item(), global_step)

	avg_train_loss = train_loss_sum / len(train_loader)
	writer.add_scalar('Train/Avg_Loss', avg_train_loss, epoch)

	model.eval()
	test_loss = 0.0
	correct = 0
	total_tokens = 0

	test_pbar = tqdm(test_loader, desc=f'Test Epoch  {epoch}/{start_epoch+epochs-1}', leave=False)
	with torch.no_grad():
		for data, target, true_len in test_pbar:
			data, target, true_len = data.to(device), target.to(device), true_len.cpu()

			# 计算验证 loss
			output = model(data,target,true_len)
			loss = criterion(output[:, 1:, :].permute(0,2,1), target[:, 1:])
			test_loss += loss.item()

			# 计算准确率（使用 generate 逐步推理）
			pred_ids = model.generate(data,1,2,x_true_len=true_len,max_len=target.size(1))  # [bs, seq_len]

			seq_len = min(pred_ids.size(1), target.size(1) - 1)
			pred_ids = pred_ids[:, :seq_len]
			target_trim = target[:, 1:seq_len+1]

			# 准确率统计（忽略 padding）
			mask = (target_trim != 0)
			correct += pred_ids.eq(target_trim).masked_select(mask).sum().item()
			total_tokens += mask.sum().item()

			test_pbar.set_postfix({'loss': f'{loss.item():.4f}','acc': f'{100.*correct/total_tokens:.2f}%'})

	# 计算平均测试损失和准确率（修正了原代码中的逻辑）
	avg_test_loss = test_loss / len(test_loader)
	accuracy = 100. * correct / total_tokens

	# 将测试指标记录到 TensorBoard
	writer.add_scalar('Test/Epoch_Loss', avg_test_loss, epoch)
	writer.add_scalar('Test/Accuracy', accuracy, epoch)

	print(f'Epoch {epoch} 完成 | Test Avg Loss: {avg_test_loss:.4f} | Accuracy: {correct}/{total_tokens} ({accuracy:.2f}%)')

	checkpoint = {
		'epoch': epoch,
		'model': model.state_dict(),
		'optimizer': optimizer.state_dict(),
		'best_acc': best_acc
	}
	print("保存本轮最后一个模型")
	torch.save(checkpoint, last_model_path)

	if accuracy > best_acc:
		print(f'>>> 发现新的最佳准确率: {accuracy:.2f}% (前最佳: {best_acc:.2f}%)，正在保存最佳模型...\n')
		best_acc = accuracy
		checkpoint['best_acc'] = best_acc  # 更新 checkpoint 里的最佳记录
		torch.save(checkpoint, best_model_path)
	else:
		print() # 输出空行以便于阅读

	scheduler.step(avg_test_loss)

writer.close()
print("训练全部完成！")



Train Epoch 0/4:   0%|          | 0/8 [00:00<?, ?it/s]

Test Epoch  0/4:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 0 完成 | Test Avg Loss: 7.4095 | Accuracy: 0/100 (0.00%)
保存本轮最后一个模型



Train Epoch 1/4:   0%|          | 0/8 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [77]:
def text2id(text,max_len):
	text_id = []
	true_len = []
	for sentence in text:
		true_len.append(len(sentence)+2)
		temp = [char2id[char] if char in char2id else char2id['<unk>'] for char in sentence]
		temp = [char2id["<s>"]] + temp + [char2id['</s>']]

		if len(temp) < max_len:
			temp = temp + [0]*(max_len-len(temp))
		elif len(temp) > max_len:
			temp = temp[:max_len]
		text_id.append(temp)
	return text_id, true_len


In [142]:
def ids_to_text(ids):
	"""将 token id 列表转回文字，跳过特殊 token"""
	tokens = []
	for i in ids:
		w = id2char.get(str(i),'')
		if w in ('', '<p>', '</s>'):
			break
		if w == '<s>':
			continue
		tokens.append(w)
	return ''.join(tokens)

def predict(model,src_sentences,sos_id,eos_id,true_len,device,max_len=50):
	"""
    src_sentences: List[str]，每个字符串是一句上联（字级别，空格分隔或直接字符串均可）
    返回: List[str] 下联列表
    """
	model.eval()
	src_sentences = torch.tensor(src_sentences,dtype=torch.long).to(device)
	true_len = torch.tensor(true_len,dtype=torch.long)
	with torch.no_grad():
		result_ids = model.generate(src_sentences,sos_id,eos_id,true_len,max_len)
	result_ids = result_ids.cpu().tolist()
	return [ids_to_text(ids) for ids in result_ids]

In [143]:
sos_id = char2id['<s>']
eos_id = char2id['</s>']

# 加载最佳模型进行推理
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model'])

test_inputs = [
    "晚风摇树树还挺",
    "春风送暖入屠苏",
    "两个黄鹂鸣翠柳",
]

test_inputs_splited = [[j for j in i] for i in test_inputs]
max_len = max(len(sentence) for sentence in test_inputs_splited)
max_len +=2

test_in_id,test_true_len = text2id(test_inputs,max_len)
print(test_in_id,test_true_len)

outputs = predict(model, test_in_id, sos_id, eos_id,test_true_len,device,10)

print("\n===== 推理结果 =====")
for src, tgt in zip(test_inputs, outputs):
    print(f"上联：{src}")
    print(f"下联：{tgt}")
    print()


[[1, 487, 6, 509, 153, 153, 374, 1581, 2], [1, 7, 6, 372, 328, 118, 2447, 966, 2], [1, 158, 796, 167, 2670, 419, 235, 67, 2]] [9, 9, 9]

===== 推理结果 =====
上联：晚风摇树树还挺
下联：饫饫饫饫饫饫饫饫饫

上联：春风送暖入屠苏
下联：饫饫饫饫饫饫饫饫饫

上联：两个黄鹂鸣翠柳
下联：饫饫饫饫饫饫饫饫饫

